In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import date_format,to_timestamp,to_date, col, when
from pyspark.sql import functions as F



In [0]:
bronze_table = spark.read.table('jc_demoworkspace.datahive.bronze_airline_departures')
display(bronze_table.head(10))

In [0]:
#changing timestamp format to get only hours and minutes
silver_df = bronze_table\
    .withColumn('scheduled_departure_time', date_format(bronze_table.scheduled_departure_time, "HH:mm"))\
    .withColumn('actual_departure_time', date_format(bronze_table.actual_departure_time, "HH:mm"))\
    .withColumn('wheelsoff_time', date_format(bronze_table.wheelsoff_time, "HH:mm"))
display(silver_df.head(5))

In [0]:
#renaming schedule and actual elapsed so end users can decipher meaning easier

silver_df = silver_df\
    .withColumnRenamed('scheduled_elapsed_time_minutes', 'scheduled_flight_duration')\
    .withColumnRenamed('actual_elapsed_time_minutes', 'actual_flight_duration')
display(silver_df.head(10))


In [0]:
silver_df = silver_df\
    .withColumn('is_delayed', when(col('departure_delay_minutes') > 0, 'Yes').otherwise('No') )\
    .withColumn('total_delay_minutes', col('delay_carrier_minutes') + col('delay_weather_minutes') + col('delay_national_aviation_system_minutes') + col('delay_security_minutes') + col('delay_late_aircraft_arrival_minutes'))\
    .withColumn('primary_delay',
                F.when(col('total_delay_minutes') == 0, None).otherwise(
                    F.greatest(
                        col('delay_carrier_minutes'),
                        col('delay_weather_minutes'),
                        col('delay_national_aviation_system_minutes'),
                        col('delay_security_minutes'),
                        col('delay_late_aircraft_arrival_minutes')
                    )
                )
    )\
    .withColumn('num_of_primary_delays',sum(
                F.when(col(c)==col('primary_delay'), 1).otherwise(0)
                for c in ['delay_carrier_minutes','delay_weather_minutes','delay_national_aviation_system_minutes','delay_security_minutes', 'delay_late_aircraft_arrival_minutes']
            )
    )\
    .withColumn('primary_delay', 
                F.when( col('num_of_primary_delays') > 1, 'Multiple Primary Delays')
                .when(col('primary_delay') == col('delay_carrier_minutes'), 'Carrier')
                .when(col('primary_delay') == col('delay_weather_minutes'), 'Weather')
                .when(col('primary_delay') == col('delay_national_aviation_system_minutes'), 'NAS')
                .when(col('primary_delay') == col('delay_security_minutes'), 'Security')
                .when(col('primary_delay') == col('delay_late_aircraft_arrival_minutes'), 'Late Aircraft')
                .otherwise('No Delay')
    )
#needed to have a usecase for when there are multiple equal primary delay reasons - may have been over kill as only 11 rows returned in Feb 2025

display(silver_df.head(10))

In [0]:
silver_df_refined =\
 silver_df.filter(col('date').isNotNull())\
.drop('num_of_primary_delays')
display(silver_df_refined.head(10))

In [0]:
silver_df_refined.write.mode('overwrite').saveAsTable('jc_demoworkspace.datahive.silver_airline_departures')
spark.read.table('jc_demoworkspace.datahive.silver_airline_departures').limit(10)